In [1]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB  # Using the new multistate version
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import NLMATH
#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 29.96it/s]


Numba compilation complete!


In [ ]:
titledpath = "D:"
savedir = titledpath + "\\2025collection\\"    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "ePN3-TS-ER"
respondercsv = responder + ".csv"
wt = "w1118"

In [ ]:
lstnew=[]

if "OPN3" in responder:
    lst = ["46416jus", "vGAT", "nSyb", "OK371", "vGlut", 'elav']

if responder == "ACR [ATR]":
    lst = ["elav", "46416jus", "vGlut", "OK371"]
    lst = [ "OK371"]
    
if responder == "ACR":
    lst = ['OK371']
    
if responder == "PdCO":
    lst = ['elav']
    

print(lst)

['46416jus', 'vGAT', 'nSyb', 'elav']


## With WT samples

In [ ]:
# Process data with all three state comparisons
all_comparisons = pd.DataFrame()
comparison_types = ['DARK-FULL', 'DARK-RECOVERY']

for n in lst:
    driver = n
    print(f"Processing {n}...")
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
    
    # Calculate metrics for all states (including Recovery)
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True) 
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    
    # Process each comparison type
    for comparison in comparison_types:
        print(f"  Computing {comparison} comparison...")
        
        dfs2 = NLMATH.deltaversion_multistate(df_sp, "Velocity", "speed", comparison)
        dfh2 = NLMATH.deltaversion_multistate(df_h, "Y", "height", comparison)
        dff2_prop = NLMATH.deltaversion_binary_multistate(df_f, "binary_fallvalue", "fallprop", comparison)
        
        # Combine all metrics for this comparison
        dftotal = pd.concat([dfs2, dfh2, dff2_prop], axis = 1)
        dftotal['genotype'] = n
        dftotal['State_Comparison'] = comparison
        dftotal['comparison_type'] = comparison  # Keep for consistency
        
        #samplesize
        samplesizelst = []
        for j in dfexpt.columns.unique().tolist()[2:]:
            number = j.split("_")[-1]
            samplesizelst.append(int(number))
        samplesize = len(list(set(samplesizelst)))
        dftotal['Sample_size'] = samplesize
        
        # Save individual comparison file
        dftotal.set_index("genotype", inplace = True)
        comparison_suffix = comparison.replace('-', '_')
        dftotal.to_csv(savedir + n + " x " + responder + f" allstats_{comparison_suffix}" + "_wtcomparison.csv")
        dftotal.reset_index(inplace=True)
    
print("\nProcessing complete!")